Process the APK list table in `notmatched/expanded_apk_policy_url_with_apkid.csv`, validate each `policy_link` one by one, and generate a cleaned CSV with the new columns:

url_status
final_url
is_accessible
notes

The original table fields will be preserved as much as possible.

In [5]:
import pandas as pd
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
INPUT = "LLM_experimental_annotations/expanded_apk_policy_url_with_apkid.csv"
OUTPUT = "LLM_experimental_annotations/expanded_apk_policy_url_with_apkid_validated.csv"

df = pd.read_csv(INPUT)
df.head()

In [7]:
url_col = "policy_link"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    )
}

In [ ]:
def check_url(url: str):
    if pd.isna(url) or not str(url).strip():
        return {
            "url_status": "",
            "final_url": "",
            "is_accessible": False,
            "notes": "empty url",
        }

    url = str(url).strip()
    session = requests.Session()
    session.headers.update(HEADERS)

    try:
        # GET is more stable than HEAD; many sites block HEAD requests.
        r = session.get(url, allow_redirects=True, timeout=15)
        final_url = r.url
        status = r.status_code
        ctype = r.headers.get("Content-Type", "")
        text_sample = ""
        if "html" in ctype.lower() or "text" in ctype.lower():
            text_sample = r.text[:800].lower()

        notes = []
        accessible = 200 <= status < 400

        if 300 <= status < 400:
            notes.append("redirect")
        if status >= 400:
            notes.append("http_error")
        if "captcha" in text_sample or "access denied" in text_sample or "forbidden" in text_sample:
            notes.append("possible_blocking")
        if accessible and not notes:
            notes.append("ok")

        return {
            "url_status": status,
            "final_url": final_url,
            "is_accessible": accessible,
            "notes": "; ".join(notes),
        }

    except requests.exceptions.SSLError:
        return {
            "url_status": "ssl_error",
            "final_url": "",
            "is_accessible": False,
            "notes": "ssl error",
        }
    except requests.exceptions.TooManyRedirects:
        return {
            "url_status": "too_many_redirects",
            "final_url": "",
            "is_accessible": False,
            "notes": "too many redirects",
        }
    except requests.exceptions.Timeout:
        return {
            "url_status": "timeout",
            "final_url": "",
            "is_accessible": False,
            "notes": "timeout",
        }
    except requests.exceptions.RequestException as e:
        return {
            "url_status": "request_error",
            "final_url": "",
            "is_accessible": False,
            "notes": str(e)[:200],
        }

unique_urls = df[url_col].dropna().astype(str).str.strip().unique().tolist()
results = {}

with ThreadPoolExecutor(max_workers=16) as ex:
    future_map = {ex.submit(check_url, u): u for u in unique_urls}
    for i, fut in enumerate(as_completed(future_map), 1):
        u = future_map[fut]
        try:
            results[u] = fut.result()
        except Exception as e:
            results[u] = {
                "url_status": "internal_error",
                "final_url": "",
                "is_accessible": False,
                "notes": str(e)[:200],
            }
        if i % 50 == 0:
            print(f"checked {i}/{len(unique_urls)}")

df["url_status"] = df[url_col].astype(str).str.strip().map(lambda u: results.get(u, {}).get("url_status", ""))
df["final_url"] = df[url_col].astype(str).str.strip().map(lambda u: results.get(u, {}).get("final_url", ""))
df["is_accessible"] = df[url_col].astype(str).str.strip().map(lambda u: results.get(u, {}).get("is_accessible", False))
df["notes"] = df[url_col].astype(str).str.strip().map(lambda u: results.get(u, {}).get("notes", ""))
df["url_domain"] = df[url_col].astype(str).str.strip().map(
    lambda u: urlparse(u).netloc if u and u != "nan" else ""
)

df.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
print(f"saved: {OUTPUT}")
print(df["is_accessible"].value_counts(dropna=False))

checked 50/429
checked 100/429
checked 150/429
checked 200/429
checked 250/429
checked 300/429
checked 350/429
checked 400/429
saved: LLM_experimental_annotations/expanded_apk_policy_url_with_apkid_validated.csv
is_accessible
True     521
False    129
Name: count, dtype: int64


Extract the rows where the `pp url` is `true` from `LLM_experimental_annotations/expanded_apk_policy_url_with_apkid_validated.csv`, then deduplicate by privacy policy URL, and finally iterate through each APK's privacy policy URL and download it as Markdown, naming each file with the APK name. Save each file under the `out_markdown` directory.

In [2]:
import os
import re
import time
import pandas as pd
import requests

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
# ====== Path configuration ======
CSV_PATH = "LLM_experimental_annotations/expanded_apk_policy_url_with_apkid_validated.csv"
OUT_DIR = "LLM_experimental_annotations/out_markdown"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    )
}

def is_true(v):
    if pd.isna(v):
        return False
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    return s in {"true", "1", "yes", "y", "t"}

def to_bool(v):
    if pd.isna(v):
        return False
    return str(v).strip().lower() in {"true", "1", "yes", "y", "t"}

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def safe_filename(name, fallback):
    name = str(name or "").strip()
    if not name:
        name = fallback
    name = re.sub(r'[\\/:*?"<>|]', "_", name)  # Replace Windows-invalid filename characters
    name = re.sub(r"\s+", " ", name).strip()
    return name[:120] if name else fallback

In [5]:
df = pd.read_csv(CSV_PATH)
df.head(), df.shape

(               apkname                             policy_name  \
 0            1Password                1Password Privacy Policy   
 1           A101 Extra  A101 Extra Kişisel Verilerin Korunması   
 2  ACTECON app/service           Privacy Policy & Terms of Use   
 3                 AJet                   AJet Aydınlatma Metni   
 4      Absher Business                   Absher Privacy Notice   
 
                                          policy_link                    apkid  \
 0                https://1password.com/legal/privacy  com.onepassword.android   
 1  https://www.a101extra.com/sozlesmeler/kisisel-...           com.a101kapida   
 2  https://www.actecon.com/en/privacy-policy-term...                      NaN   
 3      https://ajet.com/tr/kurumsal/aydinlatma-metni                 com.ajet   
 4  https://www.absher.sa/wps/vanityurl/en/privacy...     sa.gov.moi.ebusiness   
 
   apkid_confidence                   ccpa_relevant_text_or_paraphrase  \
 0             high  说明会收集账户

In [ ]:
# 1) filtered by is_accessible == true
rows = df[df["is_accessible"].map(is_true)].copy()
rows.head(), rows.shape

(                  apkname                    policy_name  \
 0               1Password       1Password Privacy Policy   
 2     ACTECON app/service  Privacy Policy & Terms of Use   
 6   Acronis mobile app(s)      Acronis Privacy Statement   
 7          ActionCard app      ActionCard Privacy Policy   
 8  Adaptavist app/service         Turkish Privacy Policy   
 
                                          policy_link                    apkid  \
 0                https://1password.com/legal/privacy  com.onepassword.android   
 2  https://www.actecon.com/en/privacy-policy-term...                      NaN   
 6        https://www.acronis.com/en/company/privacy/                      NaN   
 7                 https://actioncardapp.com/privacy/                      NaN   
 8  https://www.adaptavist.com/tr-tr/gizlilik-poli...                      NaN   
 
   apkid_confidence                   ccpa_relevant_text_or_paraphrase  \
 0             high  说明会收集账户、计费、支持、设备和服务使用数据；同时强调保险库内容的加密隔离，体现最小

In [ ]:
# # 2) policy_link filter out empty or "nan" values
# rows["policy_link"] = rows["policy_link"].astype(str).str.strip()
# rows = rows[rows["policy_link"].ne("") & rows["policy_link"].ne("nan")].copy()
# rows.head(), rows.shape

In [ ]:
# 3) filter based on policy_link, keeping the first occurrence
rows = rows.drop_duplicates(subset=["policy_link"], keep="first").copy()
rows.head(), rows.shape

(                  apkname                    policy_name  \
 0               1Password       1Password Privacy Policy   
 2     ACTECON app/service  Privacy Policy & Terms of Use   
 6   Acronis mobile app(s)      Acronis Privacy Statement   
 7          ActionCard app      ActionCard Privacy Policy   
 8  Adaptavist app/service         Turkish Privacy Policy   
 
                                          policy_link                    apkid  \
 0                https://1password.com/legal/privacy  com.onepassword.android   
 2  https://www.actecon.com/en/privacy-policy-term...                      NaN   
 6        https://www.acronis.com/en/company/privacy/                      NaN   
 7                 https://actioncardapp.com/privacy/                      NaN   
 8  https://www.adaptavist.com/tr-tr/gizlilik-poli...                      NaN   
 
   apkid_confidence                   ccpa_relevant_text_or_paraphrase  \
 0             high  说明会收集账户、计费、支持、设备和服务使用数据；同时强调保险库内容的加密隔离，体现最小

In [ ]:
print(f"is_accessible=true before filter: {df['is_accessible'].map(is_true).sum()}")
print(f"按 policy_link after filtering: {len(rows)}")

In [ ]:
# 1) Filter rows where is_accessible == true
rows = df[df["is_accessible"].map(is_true)].copy()
rows.head(), rows.shape

# # 2) Clean policy_link values and remove empty entries
# rows["policy_link"] = rows["policy_link"].astype(str).str.strip()
# rows = rows["policy_link"].ne("") & rows["policy_link"].ne("nan")].copy()
# rows.head(), rows.shape

# 3) Deduplicate by policy_link (keep the first row)
rows = rows.drop_duplicates(subset=["policy_link"], keep="first").copy()
rows.head(), rows.shape

print(f"Rows with is_accessible=true before deduplication: {df['is_accessible'].map(is_true).sum()}")
print(f"After deduplication by policy_link: {len(rows)}")

# html -> markdown
try:
    import html2text
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "html2text"])
    import html2text

converter = html2text.HTML2Text()
converter.ignore_links = False
converter.ignore_images = False
converter.body_width = 0

saved = 0
used_names = set()

# 4) Iterate through each deduplicated apk + policy_link, download and save as markdown
for idx, row in rows.iterrows():
    url = row["policy_link"]
    apkname = safe_filename(row.get("apkname", ""), fallback=f"apk_{idx}")

    filename = f"{apkname}.md"
    n = 2
    while filename in used_names or os.path.exists(os.path.join(OUT_DIR, filename)):
        filename = f"{apkname}__{n}.md"
        n += 1

    out_path = os.path.join(OUT_DIR, filename)

    try:
        r = requests.get(url, headers=HEADERS, timeout=25, allow_redirects=True)
        if r.status_code >= 400:
            print(f"[SKIP] HTTP {r.status_code} | {url}")
            continue

        md_text = converter.handle(r.text)
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        used_names.add(filename)
        saved += 1
        print(f"[OK] {filename}")
        time.sleep(0.2)
    except Exception as e:
        print(f"[ERR] {apkname} | {url} | {e}")

print(f"Completed: saved {saved} markdown files to {OUT_DIR}")

Clean the markdown files downloaded under `LLM_experimental_annotations\out_markdown` as follows:

1. Remove markdown files whose content is shorter than 10 words.
2. Remove markdown files that appear to be garbled.
3. Detect files that are not in English.
4. Save the remaining results.

In [14]:
from pathlib import Path
import shutil
import csv
import re
import unicodedata

In [ ]:
# =========================
# Configuration
# =========================
ROOT_DIR = Path(r"LLM_experimental_annotations\out_markdown")

# when False, move to removed subdirectory for safety; when True, delete directly
# False -> only move to the removed subdirectory for safety; 
# True -> delete directly
DELETE_DIRECTLY = False

# For the condition of "less than 10," the count is based on the number of words.
MIN_WORDS = 10

# output log
LOG_CSV = ROOT_DIR / "cleaning_log.csv"
SUMMARY_TXT = ROOT_DIR / "cleaning_summary.txt"

#  The directory from which the files were moved
REMOVED_DIR = ROOT_DIR / "_removed"
REMOVED_TOO_SHORT = REMOVED_DIR / "too_short"
REMOVED_GARBLED = REMOVED_DIR / "garbled"
REMOVED_NON_ENGLISH = REMOVED_DIR / "non_english"

In [ ]:
from pathlib import Path
import shutil
import csv
import re
import unicodedata

# =========================
# Configuration
# =========================
ROOT_DIR = Path(r"LLM_experimental_annotations\out_markdown")

# Whether to actually delete files:
# False -> move to the removed subdirectory for safety
# True  -> delete immediately
DELETE_DIRECTLY = False

# Less than 10 words is considered too short; this is based on word count.
MIN_WORDS = 10

# Output log
LOG_CSV = ROOT_DIR / "cleaning_log.csv"
SUMMARY_TXT = ROOT_DIR / "cleaning_summary.txt"

# Directory for moved files
REMOVED_DIR = ROOT_DIR / "_removed"
REMOVED_TOO_SHORT = REMOVED_DIR / "too_short"
REMOVED_GARBLED = REMOVED_DIR / "garbled"
REMOVED_NON_ENGLISH = REMOVED_DIR / "non_english"


# =========================
# Basic utilities
# =========================
def read_text_safe(path: Path):
    """
    Try to read the text using multiple encodings.
    Returns: (text, encoding_used, error_msg)
    """
    encodings = ["utf-8", "utf-8-sig", "gb18030", "latin-1"]
    last_error = None

    for enc in encodings:
        try:
            text = path.read_text(encoding=enc, errors="strict")
            return text, enc, ""
        except Exception as e:
            last_error = str(e)

    # Final fallback: try to recover as much as possible
    try:
        text = path.read_text(encoding="utf-8", errors="replace")
        return text, "utf-8-replace", f"strict decode failed, fallback used: {last_error}"
    except Exception as e:
        return "", "", f"all decode failed: {e}"


def strip_markdown_noise(text: str) -> str:
    """
    Roughly remove markdown symbols that add noise for statistics.
    This is a lightweight cleanup, not a full markdown parser.
    """
    t = text

    # Remove fenced code blocks
    t = re.sub(r"```.*?```", " ", t, flags=re.DOTALL)
    t = re.sub(r"`[^`\n]*`", " ", t)

    # Images / links
    t = re.sub(r"!\[.*?\]\(.*?\)", " ", t)
    t = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", t)

    # Titles, quotes, list markers
    t = re.sub(r"^[>\-\*\+#\s]+", "", t, flags=re.MULTILINE)

    # Replace table separators with spaces
    t = t.replace("|", " ")

    # Compress repeated whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t


def count_words(text: str) -> int:
    """
    Count English-style words.
    """
    words = re.findall(r"\b[a-zA-Z]+(?:'[a-zA-Z]+)?\b", text)
    return len(words)


def printable_ratio(text: str) -> float:
    if not text:
        return 0.0
    printable = sum(ch.isprintable() or ch in "\n\r\t" for ch in text)
    return printable / len(text)


def replacement_char_ratio(text: str) -> float:
    if not text:
        return 0.0
    bad = text.count("�") + text.count("\x00")
    return bad / len(text)


def weird_char_ratio(text: str) -> float:
    """
    Count the proportion of abnormal characters:
    - control characters (except \n \r \t)
    - Unicode category C*
    """
    if not text:
        return 0.0

    weird = 0
    for ch in text:
        if ch in "\n\r\t":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C"):
            weird += 1
    return weird / len(text)


def english_like_ratio(text: str) -> float:
    """
    Estimate the proportion of text that looks like English words.
    """
    tokens = re.findall(r"\b[\w'-]+\b", text)
    if not tokens:
        return 0.0

    english_like = 0
    for tok in tokens:
        if re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", tok):
            english_like += 1

    return english_like / len(tokens)


def non_ascii_ratio(text: str) -> float:
    if not text:
        return 0.0
    return sum(ord(ch) > 127 for ch in text) / len(text)


def is_garbled(text: str) -> tuple[bool, str]:
    """
    Check whether the text is likely garbled.
    Returns: (is_garbled, reason)
    """
    if not text.strip():
        return True, "empty_after_read"

    rep_ratio = replacement_char_ratio(text)
    weird_ratio = weird_char_ratio(text)
    pr_ratio = printable_ratio(text)
    eng_ratio = english_like_ratio(text)

    # Common strong garbled-text signals
    if rep_ratio > 0.01:
        return True, f"replacement_char_ratio={rep_ratio:.4f}"

    if weird_ratio > 0.02:
        return True, f"weird_char_ratio={weird_ratio:.4f}"

    if pr_ratio < 0.85:
        return True, f"printable_ratio={pr_ratio:.4f}"

    # If the text looks like it has almost no normal English tokens while not being clearly another natural language,
    # it may be a garbled extraction.
    cleaned = strip_markdown_noise(text)
    if len(cleaned) >= 100:
        token_count = len(re.findall(r"\b[\w'-]+\b", cleaned))
        if token_count >= 20 and eng_ratio < 0.20 and non_ascii_ratio(cleaned) < 0.30:
            return True, f"english_like_ratio_too_low={eng_ratio:.4f}"

    return False, ""


def is_non_english(text: str) -> tuple[bool, str]:
    """
    Roughly determine whether the text is not English.
    This does not rely on third-party libraries and uses a conservative rule set.

    Idea:
    1. Extract the cleaned text
    2. Measure the English word ratio
    3. If the non-ASCII ratio is high and the English word ratio is low, treat it as non-English
    """
    cleaned = strip_markdown_noise(text)
    if not cleaned:
        return True, "empty_after_clean"

    words = re.findall(r"\b[\w'-]+\b", cleaned)
    if len(words) < 5:
        # Very short text is hard to classify; leave it to the upstream too_short rule.
        return False, "too_short_for_language_detection"

    english_words = re.findall(r"\b[A-Za-z]+(?:'[A-Za-z]+)?\b", cleaned)

    word_ratio = len(english_words) / max(len(words), 1)
    ascii_letter_count = sum(("A" <= ch <= "Z") or ("a" <= ch <= "z") for ch in cleaned)
    letter_count = sum(ch.isalpha() for ch in cleaned)
    ascii_letter_ratio = ascii_letter_count / max(letter_count, 1)
    na_ratio = non_ascii_ratio(cleaned)

    # Empirical rules; adjust as needed for the dataset.
    # In English markdown, the proportion of ASCII letters and English words should be noticeably higher.
    if word_ratio < 0.60 and ascii_letter_ratio < 0.70 and na_ratio > 0.20:
        return True, (
            f"word_ratio={word_ratio:.4f}, "
            f"ascii_letter_ratio={ascii_letter_ratio:.4f}, "
            f"non_ascii_ratio={na_ratio:.4f}"
        )

    return False, (
        f"word_ratio={word_ratio:.4f}, "
        f"ascii_letter_ratio={ascii_letter_ratio:.4f}, "
        f"non_ascii_ratio={na_ratio:.4f}"
    )


def ensure_dirs():
    if not DELETE_DIRECTLY:
        REMOVED_TOO_SHORT.mkdir(parents=True, exist_ok=True)
        REMOVED_GARBLED.mkdir(parents=True, exist_ok=True)
        REMOVED_NON_ENGLISH.mkdir(parents=True, exist_ok=True)


def move_or_delete(path: Path, reason: str):
    if DELETE_DIRECTLY:
        path.unlink(missing_ok=True)
        return "deleted"

    mapping = {
        "too_short": REMOVED_TOO_SHORT,
        "garbled": REMOVED_GARBLED,
        "non_english": REMOVED_NON_ENGLISH,
    }
    target_dir = mapping[reason]
    target_path = target_dir / path.name

    # Handle name collisions
    if target_path.exists():
        stem = path.stem
        suffix = path.suffix
        i = 1
        while True:
            new_target = target_dir / f"{stem}__dup{i}{suffix}"
            if not new_target.exists():
                target_path = new_target
                break
            i += 1

    shutil.move(str(path), str(target_path))
    return f"moved_to:{target_path}"


# =========================
# Main process
# =========================
def clean_markdown_dir(root_dir: Path):
    if not root_dir.exists():
        raise FileNotFoundError(f"Directory does not exist: {root_dir}")

    ensure_dirs()

    md_files = sorted(root_dir.glob("*.md"))
    # If recursive traversal is needed, replace the line above with:
    # md_files = sorted(root_dir.rglob("*.md"))

    logs = []

    summary = {
        "total_files": 0,
        "kept": 0,
        "removed_too_short": 0,
        "removed_garbled": 0,
        "removed_non_english": 0,
        "read_error": 0,
    }

    for path in md_files:
        # Skip the log files themselves
        if path.name in {LOG_CSV.name, SUMMARY_TXT.name}:
            continue

        summary["total_files"] += 1

        raw_text, encoding_used, read_error = read_text_safe(path)
        cleaned = strip_markdown_noise(raw_text)
        char_count = len(cleaned)
        word_count = count_words(cleaned)

        record = {
            "file_name": path.name,
            "file_path": str(path),
            "status": "kept",
            "reason": "ok",
            "action": "",
            "encoding": encoding_used,
            "char_count": char_count,
            "word_count": word_count,
            "details": "",
            "read_error": read_error,
        }

        # 1) Remove files with fewer than 10 words
        if word_count < MIN_WORDS:
            action = move_or_delete(path, "too_short")
            record["status"] = "removed"
            record["reason"] = "too_short"
            record["action"] = action
            record["details"] = f"word_count={word_count} < {MIN_WORDS}"
            summary["removed_too_short"] += 1
            logs.append(record)
            continue

        # 2) Detect garbled content
        garbled, garbled_reason = is_garbled(raw_text)
        if garbled:
            action = move_or_delete(path, "garbled")
            record["status"] = "removed"
            record["reason"] = "garbled"
            record["action"] = action
            record["details"] = garbled_reason
            summary["removed_garbled"] += 1
            logs.append(record)
            continue

        # 3) Detect non-English content
        non_en, lang_reason = is_non_english(raw_text)
        if non_en:
            action = move_or_delete(path, "non_english")
            record["status"] = "removed"
            record["reason"] = "non_english"
            record["action"] = action
            record["details"] = lang_reason
            summary["removed_non_english"] += 1
            logs.append(record)
            continue

        # Keep the file
        record["action"] = "kept"
        record["details"] = lang_reason if 'lang_reason' in locals() else ""
        summary["kept"] += 1
        logs.append(record)

    # Write the log CSV
    with LOG_CSV.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "file_name",
                "file_path",
                "status",
                "reason",
                "action",
                "encoding",
                "char_count",
                "word_count",
                "details",
                "read_error",
            ],
        )
        writer.writeheader()
        writer.writerows(logs)

    # Write the summary report
    with SUMMARY_TXT.open("w", encoding="utf-8") as f:
        for k, v in summary.items():
            f.write(f"{k}: {v}\n")

    print("Cleaning complete")
    print(f"Log: {LOG_CSV}")
    print(f"Summary: {SUMMARY_TXT}")
    print(summary)


if __name__ == "__main__":
    clean_markdown_dir(ROOT_DIR)

In [ ]:
if __name__ == "__main__":
    clean_markdown_dir(ROOT_DIR)

Traverse the first-level subdirectories under `LLM_experimental_annotations\out_markdown` and process the markdown text in those directories without recursively traversing files inside subfolders. Organize a splitting strategy: decide how to split a very long markdown file into several parts so that each part is at most 1000 words. Here is the idea: first, do not write code yet. After reviewing the markdown, it seems splitting is not necessary.